# ToysAD-8K Post-freeze Full-pool Sensitivity Evaluation

**Notebook file:** `11b-toysad8k-full-pool-sensitivity-evaluation.ipynb`

This optional evaluator opens the separate full-pool labels and geometry only after the Notebook 11A prediction lock is verified. It describes how the unchanged frozen route behaves across all 4,107 eligible training scenes. It does not select a model, threshold, prompt or policy and does not alter the primary or official results.

## Evaluation contract

- The Notebook 11R release receipt links the analysis to the completed Notebook 10B result.
- The Notebook 11A receipt links the exact model package, method, policy and prediction lock.
- Correspondence between predicted and reference objects is computed only inside this evaluator.
- All failures remain in the denominators.
- Results are reported for the frozen 2, 3 and 5 view contracts with clustered confidence intervals, category and subgroup tables, reliability, pipeline attrition and error taxonomy.
- Evaluator overlays and masked-object panels remain review-only. ClearML receives aggregate tables and charts only.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "11b-toysad8k-full-pool-sensitivity-evaluation.ipynb"
OUTPUT_ROOT = Path(os.getenv("VPI_OUTPUT_ROOT", "/kaggle/working/toysad8k_full_pool_sensitivity_evaluation")).expanduser().resolve()
CONTRACT_PATH = OUTPUT_ROOT.parent / "evaluation_contract_11b.json"

In [ ]:
def kaggle_roots() -> list[Path]:
    root = Path("/kaggle/input")
    return sorted((path for path in root.iterdir() if path.is_dir()), key=lambda path: path.as_posix()) if root.is_dir() else []


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        for resolved in (candidate, candidate / "code"):
            if (resolved / "scripts/run_full_pool_release.py").is_file():
                return resolved.resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, variable: str) -> Path:
    configured = os.getenv(variable)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_root_with(marker: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend(kaggle_roots())
    matches = []
    for root in candidates:
        if (root / marker).is_file():
            matches.append(root.resolve())
        matches.extend(path.parent.resolve() for path in root.rglob(marker) if path.is_file())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one mounted root containing {marker}, found {len(unique)}")
    return unique[0]


def find_model_input(cohort_id: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else kaggle_roots()
    matches = []
    for root in candidates:
        records = [root / "package_record.json"] if (root / "package_record.json").is_file() else []
        records.extend(path for path in root.rglob("package_record.json") if path.is_file())
        for record in records:
            try:
                value = json.loads(record.read_text(encoding="utf-8"))
            except Exception:
                continue
            if value.get("cohort_id") == cohort_id:
                matches.append(record.parent.resolve())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one model package for {cohort_id}, found {len(unique)}")
    return unique[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source if not existing else source + os.pathsep + existing
    return environment


def clearml_mode(environment: dict[str, str]) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        access = secrets.get_secret("CLEARML_API_ACCESS_KEY")
        secret = secrets.get_secret("CLEARML_API_SECRET_KEY")
    except Exception:
        access = os.getenv("CLEARML_API_ACCESS_KEY")
        secret = os.getenv("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML setup was unavailable: {type(error).__name__}")
        return "disabled"
    return "online"

## Locate the locked prediction and evaluator resources

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/post-freeze-sensitivity/11b-toysad8k-full-pool-sensitivity-evaluation.ipynb",
)
RELEASE_RECEIPT = find_named_file("full_pool_release_receipt.json", "VPI_RELEASE_RECEIPT")
INFERENCE_RECEIPT = find_named_file("full_pool_inference_receipt.json", "VPI_INFERENCE_RECEIPT")

prediction_roots = []
evaluator_roots = []
for root in kaggle_roots():
    for path in root.rglob("prediction_lock.json"):
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if value.get("cohort_id") == "toysad8k-eligible-train-full-pool-4107-v1":
            prediction_roots.append(path.parent.parent.resolve())
    for path in root.rglob("evaluator_manifest.json"):
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if value.get("cohort_id") == "toysad8k-eligible-train-full-pool-4107-v1":
            evaluator_roots.append(path.parent.resolve())
if len(set(prediction_roots)) != 1 or len(set(evaluator_roots)) != 1:
    raise ValueError("Expected one locked prediction resource and one evaluator resource")
MOUNT_ROOTS = sorted(set([*kaggle_roots(), prediction_roots[0], evaluator_roots[0]]))
display({"prediction resource located": True, "evaluator resource located": True})

## Lock the evaluator inputs before opening labels

In [ ]:
common = [
    "--release-receipt", str(RELEASE_RECEIPT),
    "--inference-receipt", str(INFERENCE_RECEIPT),
    *sum((["--mount-root", str(root)] for root in MOUNT_ROOTS), []),
]
subprocess.run([
    sys.executable,
    str(CODE_ROOT / "scripts/run_full_pool_evaluation.py"),
    "contract",
    *common,
    "--output", str(CONTRACT_PATH),
], check=True, env=code_environment())
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
display({"contract locked": contract["status"], "labels parsed during contract creation": contract["label_files_parsed_during_contract_creation"]})

## Run the descriptive sensitivity evaluation

In [ ]:
subprocess.run([
    sys.executable,
    str(CODE_ROOT / "scripts/run_full_pool_evaluation.py"),
    "evaluate",
    *common,
    "--evaluation-contract", str(CONTRACT_PATH),
    "--output-root", str(OUTPUT_ROOT),
    "--bootstrap-repetitions", os.getenv("VPI_BOOTSTRAP_REPETITIONS", "10000"),
    "--panel-count", os.getenv("VPI_PANEL_COUNT", "16"),
], check=True, env=code_environment())

## Review the aggregate metrics and failure evidence

In [ ]:
safe_root = OUTPUT_ROOT / "safe"
for name in (
    "summary_metrics.csv",
    "clustered_confidence_intervals.csv",
    "subgroup_and_category_metrics.csv",
    "pipeline_attrition.csv",
    "prediction_failures.csv",
    "error_taxonomy.csv",
):
    table = pd.read_csv(safe_root / name)
    display(name, table.head(30))
for path in sorted((safe_root / "figures").glob("*.png")):
    display(NotebookImage(filename=str(path)))

## Review qualitative evaluator evidence

In [ ]:
inventory = json.loads((OUTPUT_ROOT / "review-only/qualitative_panel_inventory.json").read_text(encoding="utf-8"))
display({"review-only panel count": inventory["panel_count"]})
for path in sorted((OUTPUT_ROOT / "review-only").rglob("*.png"))[:16]:
    display(NotebookImage(filename=str(path)))

## Send aggregate evidence to ClearML

In [ ]:
environment = code_environment()
tracking_mode = clearml_mode(environment)
subprocess.run([
    sys.executable,
    str(CODE_ROOT / "scripts/run_full_pool_evaluation.py"),
    "track",
    "--result-root", str(OUTPUT_ROOT),
    "--mode", tracking_mode,
], check=True, env=environment)

## Interpretation boundary

Notebook 11B is a post-freeze sensitivity description. Its tables may show where the fixed route is stronger or weaker across the eligible pool, but they do not supersede Notebook 09D or Notebook 10B and cannot be used to revise the method. Any later alternative is a new study with a new protocol.